# Feature Extraction and Clustering
The goal for this notebook is to compare clustering with feature extraction vs none feature extraction to see if there are areas of similarity between disease types. This should be true as there are eyes with multiple diseases. This notebook will be testing the NMF feature extraction and using Spectral Clustering for the extracted features and images.  

A number of different hyperparameters for NMF and spectral clustering will be tested.

**N.B. By using this notebook you are expected to already have the data downloaded - See the Preprocessing Notebook.**

In [1]:
from skimage import io, color, exposure, img_as_float32
import skimage.transform
import numpy as np
import sklearn.preprocessing
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.decomposition import NMF
from sklearn.cluster import SpectralClustering
import seaborn as sns
from sklearn.metrics import silhouette_score

%matplotlib inline

In [2]:
path_to_images = "../ocular-disease-recognition-odir5k/"

training_path = "ODIR-5k/ODIR-5k/Training Images"
test_path = "ODIR-5k/ODIR-5k/Testing Images"
preprocessed_images = "preprocessed_images"

print("Training Images Length: ", len(os.listdir(os.path.join(path_to_images, training_path))))
print("Testing Images Length: ", len(os.listdir(os.path.join(path_to_images, test_path))))
print("Preprocessed Images Length: ", len(os.listdir(os.path.join(path_to_images, preprocessed_images))))

Training Images Length:  7000
Testing Images Length:  1000
Preprocessed Images Length:  6392


## NMF Feature Extraction
- Feature extraction will be performed for a 3 features.
- Batches will be used due to compuatational Limitations (RAM)
- Images will be read, resized to ensure uniform shape and normalized. 

For each n_components tried, a visual will be plotted and Spectral Clustering will be applied.

In [ ]:
#n_components = 3
training_images = os.listdir(path_to_images + training_path)
results = []
batch_size = 100
missed_batches = []
nmf = NMF(n_components=3, max_iter = 1500) #number of iterations to reach convergence seems excessive
#tried 200, 500 and kept getting warnings. Oddly enough Colab version worked with 1500 most of the time. 
for i in range(0, len(training_images), batch_size):
    image_names = training_images[i : i + batch_size]
    batch_images = [img_as_float32(io.imread(os.path.join(path_to_images, training_path + '/' + image_name))) 
                              for image_name in image_names] #reading the images in  batches
    batch_images = [color.rgba2rgb(image) if image.shape[-1] == 4 else image for image in batch_images ] #converting any 4 channel images to 3 channels
    batch_images = [skimage.transform.resize(image, (100, 100), anti_aliasing=True) for image in batch_images] #resizing 3 channel images to ensure uniform size
    normalized_batch_images = [exposure.rescale_intensity(image, out_range = (0,1)) for image in batch_images]
    flattened_images = np.reshape(normalized_batch_images, (batch_size, 100 * 100 * 3)) #flattening images for NMF so batch images is a 2d array
    shape = {image.shape[-1] for image in normalized_batch_images}
    if 4 in shape:
        missed_batches.append(i)
        continue
    else:
        batch_nmf = nmf.fit_transform(flattened_images) #fitting on NMF in batches due to memory constraints
        results.append(batch_nmf)


In [ ]:
np_results = np.array(results)

In [ ]:
np_results = np.reshape(np_results, (np_results.shape[0] * np_results.shape[1], 3))
plt.scatter(np_results[:, 0], np_results[:, 1], c='g', cmap='viridis')
plt.title('Unified NMF Results Plot')
plt.xlabel('NMF Component 1')
plt.ylabel('NMF Component 2')
plt.show()

## Clustering

To cluster the NMF results above, we'll be using Spectral Clustering. We'll first start with 8 clusters for 8 labels and then try different numbers going downwards.

<a id = "8ClustersNMF"></a>

In [ ]:
spectral_model = SpectralClustering(n_clusters=8, affinity='nearest_neighbors')
cluster_labels = spectral_model.fit_predict(np_results)

# Plot the results
plt.scatter(np_results[:, 0], np_results[:, 1], c=cluster_labels, cmap='viridis')
plt.title('Spectral Clustering Results on NMF Features')
plt.xlabel('Spectral Feature 1')
plt.ylabel('Spectral Feature 2')
plt.show()

### Amount of images in each cluster

In [ ]:
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.bar(unique,counts)

In [ ]:
print(unique, counts)

In [ ]:
silhouette_avg = silhouette_score(np_results, cluster_labels)
print(f"The average silhouette score for the eight clusters is: {silhouette_avg}")

<a id = "3ClustersNMF"></a>

In [ ]:
spectral_model = SpectralClustering(n_clusters=4, affinity='nearest_neighbors')
cluster_labels = spectral_model.fit_predict(np_results)

# Plot the results
plt.scatter(np_results[:, 0], np_results[:, 1], c=cluster_labels, cmap='viridis')
plt.title('Spectral Clustering Results on NMF Features')
plt.xlabel('Spectral Feature 1')
plt.ylabel('Spectral Feature 2')
plt.show()

In [ ]:
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.bar(unique,counts)

In [ ]:
print(unique, counts)

In [ ]:
silhouette_avg1 = silhouette_score(np_results, cluster_labels)
print(f"The average silhouette score for the four clusters is: {silhouette_avg1}")

In [ ]:
spectral_model = SpectralClustering(n_clusters=3, affinity='nearest_neighbors')
cluster_labels = spectral_model.fit_predict(np_results)

# Plot the results
plt.scatter(np_results[:, 0], np_results[:, 1], c=cluster_labels, cmap='viridis')
plt.title('Spectral Clustering Results on NMF Features')
plt.xlabel('Spectral Feature 1')
plt.ylabel('Spectral Feature 2')
plt.show()

In [ ]:
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.bar(unique,counts)

In [ ]:
print(unique, counts)

In [ ]:
#to Calculate silhouette score for his cluster after NMF has been applied 

silhouette_avg2 = silhouette_score(np_results, cluster_labels)
print(f"The average silhouette score is: {silhouette_avg2}")


## Raw Images

Earlier we saw the clustering of features extracted from the images. Below we'll attempt to the same using  the same Agglomerative Clustering Algorithm. Again this will be done in batches due to computational Limitations. 

First we'll make a graph for each image and then pass the graph to the Spectral Clustering model. Then we'll attempt to cluster the images into 8 clusters for the 8 labels. 

In [ ]:
os.environ['OMP_NUM_THREADS'] ="1"

<a id = "8ClustersRaw"></a>

In [ ]:
training_images = os.listdir(path_to_images + training_path)
raw_results = []
batch_size = 100
missed_batches = []
raw_spec = SpectralClustering(n_clusters = 8, affinity = "nearest_neighbors") 
for i in range(0, len(training_images), batch_size):
    image_names = training_images[i : i + batch_size]
    batch_images = [img_as_float32(io.imread(os.path.join(path_to_images, training_path + '/' + image_name))) 
                              for image_name in image_names] #reading the images in  batches
    batch_images = [color.rgba2rgb(image) if image.shape[-1] == 4 else image for image in batch_images ] #converting any 4 channel images to 3 channels
    batch_images = [skimage.transform.resize(image, (100, 100), anti_aliasing=True) for image in batch_images] #resizing 3 channel images to ensure uniform size
    normalized_batch_images = [exposure.rescale_intensity(image, out_range = (0,1)) for image in batch_images]
    flattened_images = np.reshape(normalized_batch_images, (batch_size, 100 * 100 * 3)) #flattening images for NMF so batch images is a 2d array
    batch_spec = raw_spec.fit_predict(flattened_images) #fitting on NMF in batches due to memory constraints
    raw_results.extend(batch_spec)


In [ ]:
unique, counts = np.unique(raw_results, return_counts=True)
plt.bar(unique,counts)

In [ ]:
cluster_results = np.reshape(np.array(raw_results), (70,100))

# Create a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(cluster_results, cmap='viridis', cbar_kws={'label': 'Cluster Assignment'})
plt.title('Spectral Clustering Results')
plt.xlabel('Image Index')
plt.ylabel('Batch')
plt.show()

<a id = "3ClustersRaw"></a>

In [ ]:
print(unique, counts)

In [ ]:
silhouette_avg1 = silhouette_score(cluster_results, cluster_labels)
print(f"The average silhouette score for these clusters is: {silhouette_avg1}")

In [ ]:
training_images = os.listdir(path_to_images + training_path)
raw_results = []
batch_size = 100
missed_batches = []
raw_spec = SpectralClustering(n_clusters = 4, affinity = "nearest_neighbors") 
for i in range(0, len(training_images), batch_size):
    image_names = training_images[i : i + batch_size]
    batch_images = [img_as_float32(io.imread(os.path.join(path_to_images, training_path + '/' + image_name))) 
                              for image_name in image_names] #reading the images in  batches
    batch_images = [color.rgba2rgb(image) if image.shape[-1] == 4 else image for image in batch_images ] #converting any 4 channel images to 3 channels
    batch_images = [skimage.transform.resize(image, (128, 128), anti_aliasing=True) for image in batch_images] #resizing 3 channel images to ensure uniform size
    normalized_batch_images = [exposure.rescale_intensity(image, out_range = (0,1)) for image in batch_images]
    flattened_images = np.reshape(normalized_batch_images, (batch_size, 128 * 128 * 3)) #flattening images for NMF so batch images is a 2d array
    batch_spec = raw_spec.fit_predict(flattened_images) #fitting on NMF in batches due to memory constraints
    raw_results.extend(batch_spec)


In [ ]:
cluster_results = np.reshape(np.array(raw_results), (70,100))

# Create a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(cluster_results, cmap='viridis', cbar_kws={'label': 'Cluster Assignment'})
plt.title('Spectral Clustering Results')
plt.xlabel('Image Index')
plt.ylabel('Batch')
plt.show()

In [ ]:
unique, counts = np.unique(raw_results, return_counts=True)
plt.bar(unique,counts)

In [ ]:
print(unique, counts)

In [ ]:
silhouette_avg1 = silhouette_score(cluster_results, cluster_labels)
print(f"The average silhouette score for these clusters is: {silhouette_avg1}")

In [ ]:
training_images = os.listdir(path_to_images + training_path)
raw_results = []
batch_size = 100
missed_batches = []
raw_spec = SpectralClustering(n_clusters = 3, affinity = "nearest_neighbors") 
for i in range(0, len(training_images), batch_size):
    image_names = training_images[i : i + batch_size]
    batch_images = [img_as_float32(io.imread(os.path.join(path_to_images, training_path + '/' + image_name))) 
                              for image_name in image_names] #reading the images in  batches
    batch_images = [color.rgba2rgb(image) if image.shape[-1] == 4 else image for image in batch_images ] #converting any 4 channel images to 3 channels
    batch_images = [skimage.transform.resize(image, (128, 128), anti_aliasing=True) for image in batch_images] #resizing 3 channel images to ensure uniform size
    normalized_batch_images = [exposure.rescale_intensity(image, out_range = (0,1)) for image in batch_images]
    flattened_images = np.reshape(normalized_batch_images, (batch_size, 128 * 128 * 3)) #flattening images for NMF so batch images is a 2d array
    batch_spec = raw_spec.fit_predict(flattened_images) #fitting on NMF in batches due to memory constraints
    raw_results.extend(batch_spec)


In [ ]:


cluster_results3 = np.reshape(np.array(raw_results), (70,100))

# Create a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(cluster_results3, cmap='viridis', cbar_kws={'label': 'Cluster Assignment'})
plt.title('Spectral Clustering Results')
plt.xlabel('Image Index')
plt.ylabel('Batch')
plt.show()

In [ ]:
unique, counts = np.unique(raw_results, return_counts=True)
plt.bar(unique,counts)

In [ ]:
print(unique, counts)

In [ ]:
silhouette_avg3 = silhouette_score(cluster_results3, cluster_labels)
print(f"The average silhouette score for these clusters is: {silhouette_avg3}")

## Analysis

We see between the Clustering with NMF and the clustering of raw images across the same cluster that Spectral clustering is better equipped to handle clustering the extracted features than it is using the raw images. 

The clustered images roughly divide the images equally among clusters, it's quite possible given the nature of Spectral Clustering that this is due to the fact that the images are very similar (retina scans). Whereas it manages to find an imbalanced split among clusters using the information extracted via NMF.

The visuals in the plot above and [the clustering on NMF with 3 Cluster](#3ClustersNMF) demonstrate this. This is further demonstrated with the equivalent 8 Clusters on [raw images](#8ClustersRaw) and [NMF extracted features](#8ClustersNMF)

These are the number of data points corresponding to each class, (there is overlap between them). 
- N: 3245
- D: 1806
- G: 331
- C: 313
- A: 280
- H: 193
- M: 270
- O: 1200

These numbers are not balanced with make it closer to the clustering produced by [NMF extracted features](#8ClustersNMF). The overlap stated above allows us to expect the lack of clear distinction between classes, as a retina may have more than one disease making Multi-Label Classification more suitable. 

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel("Processed_data.xlsx")

In [ ]:
df.head()

In [ ]:
for i in ['N', 'D', 'G', 'C', 'A', 'H', 'M','O']:
    print(len(df.loc[df[i] == 1]))